In [1]:
import pandas as pd
import numpy as np

CURRENT_YEAR = 2026

# Load Hyundai dataset
df = pd.read_csv("data/hyundai.csv")

# Load depreciation dataset
dep_df = pd.read_csv("data/annual_dep_rate.csv")

# Clean depreciation file
dep_df["make"] = dep_df["make"].astype(str).str.strip().str.upper()
dep_df["model"] = dep_df["model"].astype(str).str.strip().str.upper()

# Hyundai model mapping
MODEL_MAPPING = {
    "hyundai-accent": "ACCENT",
    "hyundai-azera": "AZERA",
    "hyundai-creta": "CRETA",
    "hyundai-elantra": "ELANTRA",
    "hyundai-genesis": "GENESIS",
    "hyundai-grand-i10": "GRAND I10",
    "hyundai-grand-santa-fe": "GRAND SANTA FE",
    "hyundai-h1": "H1",
    "hyundai-ioniq-5": "IONIQ 5",
    "hyundai-kona": "KONA",
    "hyundai-kona-hybrid": "KONA",
    "hyundai-palisade": "PALISADE",
    "hyundai-santa-fe": "SANTA FE",
    "hyundai-sonata": "SONATA",
    "hyundai-stargazer": "STARGAZER",
    "hyundai-staria": "STARIA",
    "hyundai-tucson": "TUCSON",
    "hyundai-venue": "VENUE",
    "hyundai-veracruz": "VERACRUZ",
}

df["make"] = "HYUNDAI"

df["model_name"] = (
    df["model_slug"]
    .map(MODEL_MAPPING)
    .astype(str)
    .str.upper()
)

# Build lookup
dep_lookup = (
    dep_df.groupby(["make", "model"])["annual_dep_rate"]
    .mean()
    .reset_index()
)

# Merge rates
df = df.merge(
    dep_lookup,
    left_on=["make", "model_name"],
    right_on=["make", "model"],
    how="left"
)

# Car age
df["car_age"] = CURRENT_YEAR - df["year"]
df["car_age"] = df["car_age"].clip(lower=0)

# Depreciated value
def calc_depreciated_value(row):
    rate = row["annual_dep_rate"]

    if pd.isna(rate):
        return np.nan

    age = row["car_age"]

    if age == 0:
        return round(row["price_avg_aed"], 0)

    return round(
        row["price_avg_aed"] * ((1 - rate) ** age),
        0
    )

df["depreciated_value"] = df.apply(
    calc_depreciated_value,
    axis=1
)

# Validation
print("Total Rows:", len(df))
print("Matched Rates:", df["annual_dep_rate"].notna().sum())
print("Missing Rates:", df["annual_dep_rate"].isna().sum())

# Save
df.to_csv(
    "data/hyundai_with_depreciation_annual_rate.csv",
    index=False
)

print("Saved: hyundai_with_depreciation.csv")

Total Rows: 415
Matched Rates: 415
Missing Rates: 0
Saved: hyundai_with_depreciation.csv
